# 📊 Dashboard BI — Superstore Sales 2014–2017

**Source :** [Kaggle — Sample Superstore Sales Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
**Dataset :** 9 994 commandes · 21 colonnes · 4 ans (2014–2017) · USA  
**Objectif :** Dashboard de pilotage commercial complet — CA, marges, tendances, segments, géographie

> 💡 Ce notebook est le pendant Python du dashboard Power BI disponible dans `/powerbi/`.  
> Les mesures DAX et transformations Power Query sont documentées dans `powerbi/GUIDE_POWERBI.md`.

---

## Plan
1. **Pipeline de données** — Chargement, nettoyage, feature engineering
2. **KPI Cards** — Vue d'ensemble de la performance
3. **Tendance annuelle** — Croissance CA et profit
4. **Performance par catégorie & sous-catégorie** — Rentabilité
5. **Analyse par segment client** — Consumer vs Corporate vs Home Office
6. **Analyse géographique** — Performance par région
7. **Impact des remises** — Discount vs Profit
8. **Dashboard final** — Vue synthétique 6-en-1

## 1. Pipeline de données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#2d3154', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'font.family': 'monospace'
})
P = ['#4f8ef7','#f7634f','#4fe0a0','#f7c44f','#b44ff7','#f74fb8','#4fc4f7','#f7a44f']
fmt_k = mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k')
fmt_pct = mticker.FuncFormatter(lambda x, _: f'{x:.1f}%')

# ── PIPELINE ──
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

# Parsing dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

# Feature engineering
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Quarter']       = df['Order Date'].dt.quarter
df['YearMonth']     = df['Order Date'].dt.to_period('M').astype(str)
df['Ship_Days']     = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit_Margin'] = df['Profit'] / df['Sales'] * 100
df['Is_Loss']       = df['Profit'] < 0

print(f'✅ Dataset chargé : {len(df):,} commandes · {df["Customer ID"].nunique()} clients · {df["Order ID"].nunique()} ordres')
print(f'📅 Période : {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'🗺️  Régions : {sorted(df["Region"].unique())}')
print(f'🛍️  Catégories : {sorted(df["Category"].unique())}')
print(f'\n⚠️  Commandes à perte : {df["Is_Loss"].sum()} ({df["Is_Loss"].mean()*100:.1f}%)')
df.head(3)

## 2. KPI Cards — Vue d'ensemble

In [ ]:
kpis = [
    ('CA Total',          f"${df['Sales'].sum()/1e6:.2f}M",        P[0]),
    ('Profit Total',      f"${df['Profit'].sum()/1000:.0f}k",       P[2]),
    ('Marge Moyenne',     f"{df['Profit_Margin'].mean():.1f}%",     P[3]),
    ('Nb Commandes',      f"{df['Order ID'].nunique():,}",           P[1]),
    ('Nb Clients',        f"{df['Customer ID'].nunique():,}",        P[4]),
    ('Panier Moyen',      f"${df.groupby('Order ID')['Sales'].sum().mean():.0f}", P[5]),
    ('Délai Livraison',   f"{df['Ship_Days'].mean():.1f} jours",    P[6]),
    ('Taux Commandes -',  f"{df['Is_Loss'].mean()*100:.1f}%",       P[7]),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 4))
fig.suptitle('KPIs Globaux — Superstore 2014–2017', fontsize=14, fontweight='bold', color='#e6edf3')

for ax, (label, value, color) in zip(axes.flatten(), kpis):
    ax.text(0.5, 0.62, value, transform=ax.transAxes, ha='center', va='center',
            fontsize=20, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, transform=ax.transAxes, ha='center', va='center',
            fontsize=9, color='#8b949e')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('outputs/01_kpi_cards.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 3. Tendance Annuelle — Croissance CA & Profit

In [ ]:
# -- Annuel --
annual = df.groupby('Year').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
annual['Growth_CA']  = annual['Sales'].pct_change() * 100
annual['Margin']     = annual['Profit'] / annual['Sales'] * 100
print('=== Performance annuelle ===')
print(annual.round(2).to_string(index=False))

# -- Mensuel (trend) --
monthly = df.groupby('YearMonth').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
monthly['MA3'] = monthly['Sales'].rolling(3, min_periods=1).mean()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Tendance annuelle & mensuelle', fontsize=13, fontweight='bold', color='#e6edf3')

# CA et Profit annuel (barres groupées)
x = range(len(annual))
axes[0].bar([i-0.2 for i in x], annual['Sales']/1000, width=0.38, color=P[0], label='CA (k$)', alpha=0.85)
axes[0].bar([i+0.2 for i in x], annual['Profit']/1000, width=0.38, color=P[2], label='Profit (k$)', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(annual['Year'])
axes[0].set_title('CA & Profit par année')
axes[0].set_ylabel('k$')
axes[0].legend(framealpha=0.3)
axes[0].grid(axis='y', alpha=0.3)

# Croissance YoY
axes[1].bar(annual['Year'][1:], annual['Growth_CA'][1:],
            color=[P[2] if v>0 else P[1] for v in annual['Growth_CA'][1:]])
axes[1].axhline(0, color='white', linewidth=0.8)
axes[1].set_title('Croissance CA YoY (%)')
axes[1].set_ylabel('%')
for i, (year, val) in enumerate(zip(annual['Year'][1:], annual['Growth_CA'][1:])):
    axes[1].text(year, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# Tendance mensuelle + MA3
axes[2].plot(range(len(monthly)), monthly['Sales']/1000, color=P[0], alpha=0.4, linewidth=1)
axes[2].plot(range(len(monthly)), monthly['MA3']/1000, color=P[0], linewidth=2.5, label='Moyenne mobile 3m')
axes[2].fill_between(range(len(monthly)), monthly['MA3']/1000, alpha=0.1, color=P[0])
# Repères annuels
for year in [2015, 2016, 2017]:
    idx = monthly[monthly['YearMonth'].str.startswith(str(year))].index[0]
    axes[2].axvline(idx, color='#8b949e', linestyle='--', linewidth=0.8)
    axes[2].text(idx+0.3, monthly['MA3'].max()/1000*0.95, str(year), fontsize=8, color='#8b949e')
axes[2].set_title('CA mensuel (+ moyenne mobile 3m)')
axes[2].set_ylabel('k$')
axes[2].set_xticks([])
axes[2].legend(framealpha=0.3)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_annual_trend.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 4. Performance par Catégorie & Sous-catégorie

In [ ]:
cat = df.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique')).reset_index()
cat['Margin'] = cat['Profit'] / cat['Sales'] * 100

subcat = df.groupby('Sub-Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
subcat['Margin'] = subcat['Profit'] / subcat['Sales'] * 100
subcat = subcat.sort_values('Profit', ascending=True)

print('=== Catégories ===')
print(cat.round(2).to_string(index=False))
print('\n=== Sous-catégories à perte ===')
print(subcat[subcat['Profit']<0].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Performance par catégorie & sous-catégorie', fontsize=13, fontweight='bold', color='#e6edf3')

# Catégories : bubble chart CA vs Marge
for i, row in cat.iterrows():
    axes[0].scatter(row['Sales']/1000, row['Margin'], s=row['Orders']*2,
                    color=P[i], alpha=0.8, zorder=3)
    axes[0].annotate(row['Category'], (row['Sales']/1000, row['Margin']),
                     textcoords='offset points', xytext=(8,4), fontsize=11)
axes[0].axhline(0, color='#f7634f', linestyle='--', linewidth=1)
axes[0].set_xlabel('CA (k$)')
axes[0].set_ylabel('Marge (%)')
axes[0].set_title('CA vs Marge par catégorie\n(taille = nb commandes)')
axes[0].grid(alpha=0.3)

# Sous-catégories : barres horizontales profit
colors = [P[1] if v < 0 else P[2] for v in subcat['Profit']]
axes[1].barh(subcat['Sub-Category'], subcat['Profit']/1000, color=colors)
axes[1].axvline(0, color='white', linewidth=0.8)
axes[1].set_xlabel('Profit (k$)')
axes[1].set_title('Profit par sous-catégorie\n(rouge = perte)')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/03_category_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 5. Analyse par Segment Client

In [ ]:
seg = df.groupby('Segment').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Orders=('Order ID','nunique'), Customers=('Customer ID','nunique')
).reset_index()
seg['Margin']     = seg['Profit'] / seg['Sales'] * 100
seg['CA_per_cust']= seg['Sales'] / seg['Customers']

# Evolution annuelle par segment
seg_annual = df.groupby(['Year','Segment'])['Sales'].sum().reset_index()

print('=== Performance par segment ===')
print(seg.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Analyse par segment client', fontsize=13, fontweight='bold', color='#e6edf3')

# Part de CA
axes[0].pie(seg['Sales'], labels=seg['Segment'], colors=P[:3],
            autopct='%1.1f%%', startangle=90,
            textprops={'color':'#c9d1d9','fontsize':11})
axes[0].set_title('Répartition du CA')

# Marge & CA par client
x = range(len(seg))
axes[1].bar([i-0.2 for i in x], seg['Margin'], width=0.38, color=P[3], label='Marge %')
ax2 = axes[1].twinx()
ax2.bar([i+0.2 for i in x], seg['CA_per_cust'], width=0.38, color=P[0], alpha=0.7, label='CA/client')
axes[1].set_xticks(x); axes[1].set_xticklabels(seg['Segment'], rotation=15)
axes[1].set_ylabel('Marge (%)')
ax2.set_ylabel('CA moyen par client ($)')
axes[1].set_title('Marge & CA par client')
lines1, l1 = axes[1].get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, l1+l2, framealpha=0.3, fontsize=9)
ax2.set_facecolor('#1a1d2e')
axes[1].grid(axis='y', alpha=0.3)

# Evolution annuelle par segment
for i, seg_name in enumerate(seg['Segment']):
    sub = seg_annual[seg_annual['Segment'] == seg_name]
    axes[2].plot(sub['Year'], sub['Sales']/1000, marker='o', color=P[i],
                 label=seg_name, linewidth=2)
axes[2].set_title('CA annuel par segment')
axes[2].set_ylabel('k$')
axes[2].legend(framealpha=0.3)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_segment_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 6. Analyse Géographique — Performance par Région

In [ ]:
region = df.groupby('Region').agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    Orders=('Order ID','nunique'), Customers=('Customer ID','nunique')
).reset_index()
region['Margin'] = region['Profit'] / region['Sales'] * 100
region = region.sort_values('Sales', ascending=False)

# Top états
state = df.groupby('State').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).reset_index()
state['Margin'] = state['Profit'] / state['Sales'] * 100
top_states = state.sort_values('Sales', ascending=False).head(10)
loss_states = state[state['Profit']<0].sort_values('Profit')

print('=== Régions ===')
print(region.round(2).to_string(index=False))
print('\n=== États à perte ===')
print(loss_states[['State','Sales','Profit','Margin']].round(2).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle('Analyse géographique', fontsize=13, fontweight='bold', color='#e6edf3')

# CA par région
bars = axes[0].bar(region['Region'], region['Sales']/1000, color=P[:4])
axes[0].set_title('CA par région (k$)')
axes[0].set_ylabel('k$')
for bar, val in zip(bars, region['Sales']/1000):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'${val:.0f}k', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Marge par région
colors_m = [P[2] if m > 10 else P[3] if m > 5 else P[1] for m in region['Margin']]
bars_m = axes[1].bar(region['Region'], region['Margin'], color=colors_m)
axes[1].axhline(region['Margin'].mean(), color='white', linestyle='--', linewidth=1,
                label=f'Moy: {region["Margin"].mean():.1f}%')
axes[1].set_title('Marge par région (%)')
axes[1].set_ylabel('%')
for bar, val in zip(bars_m, region['Margin']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{val:.1f}%', ha='center', fontsize=10)
axes[1].legend(framealpha=0.3)
axes[1].grid(axis='y', alpha=0.3)

# Top 10 états par CA
axes[2].barh(top_states['State'][::-1], top_states['Sales'][::-1]/1000, color=P[0])
axes[2].set_title('Top 10 états par CA')
axes[2].set_xlabel('k$')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_geographic_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. Impact des Remises sur la Rentabilité

In [ ]:
# Discrétisation des remises
df['Discount_Band'] = pd.cut(df['Discount'],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=['0%','1-10%','11-20%','21-30%','31-50%','>50%'])

disc = df.groupby('Discount_Band', observed=True).agg(
    Sales=('Sales','sum'), Profit=('Profit','sum'),
    nb_orders=('Order ID','count')
).reset_index()
disc['Margin'] = disc['Profit'] / disc['Sales'] * 100

print('=== Impact remises ===')
print(disc.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impact des remises sur la rentabilité', fontsize=13, fontweight='bold', color='#e6edf3')

# Marge par tranche de remise
colors_d = [P[2] if m > 0 else P[1] for m in disc['Margin']]
axes[0].bar(disc['Discount_Band'].astype(str), disc['Margin'], color=colors_d)
axes[0].axhline(0, color='white', linewidth=0.8)
axes[0].set_title('Marge moyenne par tranche de remise')
axes[0].set_xlabel('Remise')
axes[0].set_ylabel('Marge (%)')
for i, (val, n) in enumerate(zip(disc['Margin'], disc['nb_orders'])):
    axes[0].text(i, val + (1 if val>=0 else -3), f'{val:.1f}%\n({n} cmd)', ha='center', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Scatter Discount vs Profit_Margin (échantillon)
sample = df.sample(2000, random_state=42)
axes[1].scatter(sample['Discount'], sample['Profit_Margin'],
                c=sample['Profit_Margin'].apply(lambda x: P[1] if x<0 else P[0]),
                alpha=0.3, s=20)
axes[1].axhline(0, color='white', linewidth=1)
axes[1].axvline(0.2, color='#f7c44f', linewidth=1, linestyle='--', label='Remise 20%')
# Trend line
z = np.polyfit(sample['Discount'], sample['Profit_Margin'], 1)
p_fit = np.poly1d(z)
x_line = np.linspace(0, 0.85, 100)
axes[1].plot(x_line, p_fit(x_line), color='#f7c44f', linewidth=2, label='Tendance')
axes[1].set_xlabel('Remise appliquée')
axes[1].set_ylabel('Marge (%)')
axes[1].set_title('Remise vs Marge (échantillon 2k commandes)')
axes[1].set_ylim(-150, 100)
axes[1].legend(framealpha=0.3)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/06_discount_impact.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 8. Dashboard Final — Vue Synthétique 6-en-1

In [ ]:
fig = plt.figure(figsize=(20, 14), facecolor='#0f1117')
fig.suptitle('📊 Superstore Sales Dashboard 2014–2017',
             fontsize=20, fontweight='bold', color='#e6edf3', y=0.99)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── A : CA mensuel + MA ──
ax_a = fig.add_subplot(gs[0, :])
ax_a.set_facecolor('#1a1d2e')
ax_a.bar(range(len(monthly)), monthly['Sales']/1000, color=P[0], alpha=0.45, width=0.8)
ax_a.plot(range(len(monthly)), monthly['MA3']/1000, color=P[0], linewidth=2.5, label='MA 3m')
ax_b2 = ax_a.twinx(); ax_b2.set_facecolor('#1a1d2e')
ax_b2.plot(range(len(monthly)), monthly['Profit']/1000, color=P[2], linewidth=2, linestyle='--', label='Profit')
for year in [2015,2016,2017]:
    idx = monthly[monthly['YearMonth'].str.startswith(str(year))].index[0]
    ax_a.axvline(idx, color='#8b949e', linestyle=':', linewidth=0.8)
    ax_a.text(idx+0.4, monthly['MA3'].max()/1000*0.93, str(year), fontsize=9, color='#8b949e')
ax_a.set_ylabel('CA (k$)'); ax_b2.set_ylabel('Profit (k$)')
ax_a.set_title('CA mensuel & Profit — 2014 à 2017', fontsize=12, fontweight='bold')
ax_a.set_xticks([])
lines1,l1 = ax_a.get_legend_handles_labels(); lines2,l2 = ax_b2.get_legend_handles_labels()
ax_a.legend(lines1+lines2, l1+l2, framealpha=0.3, loc='upper left')
ax_a.grid(alpha=0.3)

# ── B : Part CA par catégorie ──
ax_b = fig.add_subplot(gs[1, 0])
ax_b.set_facecolor('#1a1d2e')
ax_b.pie(cat['Sales'], labels=cat['Category'], colors=P[:3],
         autopct='%1.1f%%', startangle=140, textprops={'color':'#c9d1d9','fontsize':10})
ax_b.set_title('Part CA par catégorie', fontsize=11, fontweight='bold')

# ── C : Profit sous-catégories ──
ax_c = fig.add_subplot(gs[1, 1])
ax_c.set_facecolor('#1a1d2e')
top_sc = subcat.sort_values('Profit')
colors_sc = [P[1] if v<0 else P[2] for v in top_sc['Profit']]
ax_c.barh(top_sc['Sub-Category'], top_sc['Profit']/1000, color=colors_sc)
ax_c.axvline(0, color='white', linewidth=0.8)
ax_c.set_title('Profit sous-catégories (k$)', fontsize=11, fontweight='bold')
ax_c.grid(axis='x', alpha=0.3)

# ── D : CA & Marge par région ──
ax_d = fig.add_subplot(gs[1, 2])
ax_d.set_facecolor('#1a1d2e')
x = range(len(region))
ax_d.bar(x, region['Sales']/1000, color=P[0], alpha=0.85, label='CA (k$)')
ax_d2 = ax_d.twinx(); ax_d2.set_facecolor('#1a1d2e')
ax_d2.plot(x, region['Margin'], color=P[3], marker='D', markersize=8, linewidth=2, label='Marge %')
ax_d.set_xticks(x); ax_d.set_xticklabels(region['Region'], rotation=20, fontsize=9)
ax_d.set_ylabel('CA (k$)'); ax_d2.set_ylabel('Marge (%)')
ax_d.set_title('CA & Marge par région', fontsize=11, fontweight='bold')
l1,la1 = ax_d.get_legend_handles_labels(); l2,la2 = ax_d2.get_legend_handles_labels()
ax_d.legend(l1+l2, la1+la2, framealpha=0.3, fontsize=9)
ax_d.grid(axis='y', alpha=0.3)

# ── E : Segment annuel ──
ax_e = fig.add_subplot(gs[2, :2])
ax_e.set_facecolor('#1a1d2e')
for i, seg_name in enumerate(['Consumer','Corporate','Home Office']):
    sub = seg_annual[seg_annual['Segment']==seg_name]
    ax_e.plot(sub['Year'], sub['Sales']/1000, marker='o', color=P[i],
              label=seg_name, linewidth=2.5, markersize=7)
    ax_e.fill_between(sub['Year'], sub['Sales']/1000, alpha=0.08, color=P[i])
ax_e.set_title('Évolution CA par segment client 2014–2017', fontsize=11, fontweight='bold')
ax_e.set_ylabel('k$')
ax_e.legend(framealpha=0.3)
ax_e.grid(alpha=0.3)

# ── F : Remise vs Marge ──
ax_f = fig.add_subplot(gs[2, 2])
ax_f.set_facecolor('#1a1d2e')
colors_d = [P[2] if m>0 else P[1] for m in disc['Margin']]
ax_f.bar(disc['Discount_Band'].astype(str), disc['Margin'], color=colors_d)
ax_f.axhline(0, color='white', linewidth=0.8)
ax_f.set_title('Marge par tranche de remise', fontsize=11, fontweight='bold')
ax_f.set_xlabel('Remise'); ax_f.set_ylabel('Marge (%)')
ax_f.grid(axis='y', alpha=0.3)

plt.savefig('outputs/07_dashboard_final.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('✅ Dashboard exporté → outputs/07_dashboard_final.png')

## 9. Synthèse & Recommandations

| KPI | Valeur | Insight |
|-----|--------|---------|
| CA Total | $2.30M | Croissance constante +51% sur 4 ans |
| Profit Total | $286k | Marge globale de 12.5% |
| Catégorie à risque | Furniture | Marge très faible (2.5%) malgré $742k de CA |
| Sous-catégorie à perte | Tables (-$17.7k) | À revoir : pricing ou politique de remise |
| Région la + rentable | West | Marge ~15% vs Central ~8% |
| Effet remise | Critique au-delà de 20% | Les remises >20% génèrent des pertes systématiques |
| Segment | Consumer dominant | 50% du CA mais marge similaire aux autres |

---

**Stack Python :** Pandas · NumPy · Matplotlib (GridSpec, double axes, scatter, trend line)  
**Power BI :** Voir `powerbi/GUIDE_POWERBI.md` pour les mesures DAX et le modèle de données